# Vault Secret Wrapping Token

Este notebook genera un wrapping token de Vault para el secreto demo `secret-wrapping/handoff`. El token resultante se entrega al supuesto receptor, que lo pega en la UI local de la demo para desenvolver el secreto una sola vez.

## 1. Set Working Directory

Cambia al directorio del módulo 8, donde están el estado local de Terraform y los outputs que necesita la demo.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
pwd

## 2. Load Terraform Outputs

Exporta `VAULT_ADDR` y `VAULT_NAMESPACE` desde el módulo 8. El `VAULT_TOKEN` se toma desde el output `admin_token` del módulo `1_Create_HCP_Vault_Cluster`, porque ese token administra el cluster de Vault.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
export VAULT_ADDR="$(terraform output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=/Users/jose/Demo/Cert_Automation_VM/1_Create_HCP_Vault_Cluster output -raw admin_token)"

vault kv get -wrap-ttl=5m -format=json secret-wrapping/handoff \
  | jq -r '.wrap_info.token'

El token que imprime la celda anterior caduca en 5 minutos y solo se puede usar una vez. Si lo usas con `curl`, ya no podrás pegar ese mismo token en la UI.

## 4. Read Wrapped Secret and Extract Token

La celda anterior solicita a Vault una respuesta envuelta con TTL de 5 minutos y extrae `.wrap_info.token`. Ese valor es el wrapping token de un solo uso que se entrega al receptor.

## 5. Verify Token Metadata

Esta celda genera otro wrapping token y muestra únicamente metadatos no sensibles. No desenvuelve el secreto.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
export VAULT_ADDR="$(terraform output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=/Users/jose/Demo/Cert_Automation_VM/1_Create_HCP_Vault_Cluster output -raw admin_token)"

vault kv get -wrap-ttl=5m -format=json secret-wrapping/handoff

## 6. Launch the Receiver Frontend

Ejecuta el servidor local de la demo y abre la app en `http://localhost:8788`. La app muestra un formulario donde el receptor pega el wrapping token y pulsa **Unwrap secret**.

La celda siguiente lanza el servidor en segundo plano para no bloquear el notebook.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
./scripts/serve.sh > /tmp/vault_wrapping_frontend.log 2>&1 &
echo "Frontend URL: http://localhost:8788"
echo "Log file: /tmp/vault_wrapping_frontend.log"

## 7. Unwrap with curl using the Vault API

Este comando genera un wrapping token nuevo y lo desenvuelve llamando directamente a la API oficial de Vault: `POST /v1/sys/wrapping/unwrap`.

La llamada usa el wrapping token como `X-Vault-Token`. Si estás en HCP Vault Dedicated, también envía `X-Vault-Namespace`.

Importante: el token queda consumido por este `curl`, así que genera otro token si después quieres probar la UI.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
export VAULT_ADDR="$(terraform output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=/Users/jose/Demo/Cert_Automation_VM/1_Create_HCP_Vault_Cluster output -raw admin_token)"

WRAP_TOKEN="$(vault kv get -wrap-ttl=5m -format=json secret-wrapping/handoff | jq -r '.wrap_info.token')"
echo "$WRAP_TOKEN" > /tmp/vault-wrap.token

echo "Wrap Token saved to /tmp/vault-wrap.token"
echo "(token value not printed — paste from the file or use the next cell to unwrap)"


In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
export VAULT_ADDR="$(terraform output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform output -raw vault_namespace)"

export WRAP_TOKEN="$(cat /tmp/vault-wrap.token)"

curl --fail --silent --show-error \
  --request POST \
  --header "X-Vault-Token: $WRAP_TOKEN" \
  --header "X-Vault-Namespace: $VAULT_NAMESPACE" \
  "$VAULT_ADDR/v1/sys/wrapping/unwrap" \
  | jq '.data'


In [ ]:
%%bash
# Second attempt to unwrap the token — must fail because it is single-use
cd /Users/jose/Demo/Cert_Automation_VM/8_Vault_Secret_Wrapping
export VAULT_ADDR="$(terraform output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform output -raw vault_namespace)"

export WRAP_TOKEN="$(cat /tmp/vault-wrap.token)"

curl --fail --silent --show-error \
  --request POST \
  --header "X-Vault-Token: $WRAP_TOKEN" \
  --header "X-Vault-Namespace: $VAULT_NAMESPACE" \
  "$VAULT_ADDR/v1/sys/wrapping/unwrap" \
  | jq '.data'

## 8. Create a Certificate with the Vault PKI Engine

El módulo 2 configura el PKI engine intermedio en `pki_int` y el role `jose-merchan-sbx-hashidemos-io`. Las siguientes celdas emiten certificados para un common name de demo bajo la zona `hosted_dns_zone` configurada en `2_PKI_Vault_VM/variables.tfvars`.

Estos ejemplos no imprimen la private key. Guardan la respuesta completa en `/tmp` para inspección local.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM
export VAULT_ADDR="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=1_Create_HCP_Vault_Cluster output -raw admin_token)"

PKI_PATH="pki_int"
PKI_ROLE="jose-merchan-sbx-hashidemos-io"
DOMAIN="$(awk -F'"' '/^hosted_dns_zone/ {print $2}' 2_PKI_Vault_VM/variables.tfvars)"
COMMON_NAME="notebook.$DOMAIN"

vault write -format=json "$PKI_PATH/issue/$PKI_ROLE" \
  common_name="$COMMON_NAME" \
  ttl="24h" \
  > /tmp/vault-pki-cli-cert.json

jq '{common_name: .data.common_name, serial_number: .data.serial_number, expiration: .data.expiration, ca_chain_length: (.data.ca_chain | length)}' /tmp/vault-pki-cli-cert.json

## 9. Create a Certificate with curl

Este ejemplo llama directamente al endpoint de Vault PKI `POST /v1/pki_int/issue/jose-merchan-sbx-hashidemos-io`. La respuesta contiene `certificate`, `private_key`, `issuing_ca`, `ca_chain`, `serial_number` y metadatos de expiración.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM
export VAULT_ADDR="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=1_Create_HCP_Vault_Cluster output -raw admin_token)"

PKI_PATH="pki_int"
PKI_ROLE="jose-merchan-sbx-hashidemos-io"
DOMAIN="$(awk -F'"' '/^hosted_dns_zone/ {print $2}' 2_PKI_Vault_VM/variables.tfvars)"
COMMON_NAME="curl-notebook.$DOMAIN"

curl --fail --silent --show-error \
  --request POST \
  --header "X-Vault-Token: $VAULT_TOKEN" \
  --header "X-Vault-Namespace: $VAULT_NAMESPACE" \
  --header "Content-Type: application/json" \
  --data "$(jq -n --arg common_name "$COMMON_NAME" --arg ttl "24h" '{common_name: $common_name, ttl: $ttl}')" \
  "$VAULT_ADDR/v1/$PKI_PATH/issue/$PKI_ROLE" \
  > /tmp/vault-pki-curl-cert.json

jq '{common_name: .data.common_name, serial_number: .data.serial_number, expiration: .data.expiration, issuing_ca_present: (.data.issuing_ca != null)}' /tmp/vault-pki-curl-cert.json

## 10. Inspect the Issued Certificate

Esta celda toma la respuesta del ejemplo con `curl`, guarda el certificado público en `/tmp/vault-pki-curl-cert.pem` y muestra subject, issuer, fechas y serial con `openssl`.

In [ ]:
%%bash
jq -r '.data.certificate' /tmp/vault-pki-curl-cert.json > /tmp/vault-pki-curl-cert.pem

openssl x509 -in /tmp/vault-pki-curl-cert.pem \
  -noout \
  -subject \
  -issuer \
  -dates \
  -serial

## 11. Create a Wrapped Certificate

Este es el flujo combinado: Vault emite un certificado con el PKI engine, pero la respuesta se entrega como wrapping token usando `-wrap-ttl`. El receptor no recibe el certificado directamente; recibe un token de un solo uso que puede desenvolver con `sys/wrapping/unwrap`.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM
export VAULT_ADDR="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_namespace)"
export VAULT_TOKEN="$(terraform -chdir=1_Create_HCP_Vault_Cluster output -raw admin_token)"

PKI_PATH="pki_int"
PKI_ROLE="jose-merchan-sbx-hashidemos-io"
DOMAIN="$(awk -F'"' '/^hosted_dns_zone/ {print $2}' 2_PKI_Vault_VM/variables.tfvars)"
COMMON_NAME="wrapped-cert.$DOMAIN"

vault write -wrap-ttl=5m -format=json "$PKI_PATH/issue/$PKI_ROLE" \
  common_name="$COMMON_NAME" \
  ttl="24h" \
  > /tmp/vault-pki-wrapped-cert-token.json

jq -r '.wrap_info.token' /tmp/vault-pki-wrapped-cert-token.json > /tmp/vault-pki-wrapped-cert.token
jq '.wrap_info | {ttl, creation_path, creation_time, wrapped_accessor: .wrapped_accessor, wrapped_token: .token}' /tmp/vault-pki-wrapped-cert-token.json

## 12. Unwrap the Certificate Response

Esta celda simula al receptor: toma el wrapping token, llama a `POST /v1/sys/wrapping/unwrap` y obtiene la respuesta original de emisión del certificado. El token queda invalidado después de este paso.

In [ ]:
%%bash
cd /Users/jose/Demo/Cert_Automation_VM
export VAULT_ADDR="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_addr)"
export VAULT_NAMESPACE="$(terraform -chdir=8_Vault_Secret_Wrapping output -raw vault_namespace)"
export WRAP_TOKEN="$(cat /tmp/vault-pki-wrapped-cert.token)"

curl --fail --silent --show-error \
  --request POST \
  --header "X-Vault-Token: $WRAP_TOKEN" \
  --header "X-Vault-Namespace: $VAULT_NAMESPACE" \
  --data '{}' \
  "$VAULT_ADDR/v1/sys/wrapping/unwrap" \
  > /tmp/vault-pki-unwrapped-cert.json

jq '{common_name: .data.common_name, serial_number: .data.serial_number, expiration: .data.expiration, private_key_type: .data.private_key_type, ca_chain_length: (.data.ca_chain | length)}' /tmp/vault-pki-unwrapped-cert.json

## 13. Inspect the Unwrapped Certificate

Guarda el certificado público de la respuesta desenvuelta y muestra sus campos principales. La private key permanece en el JSON local de `/tmp` y no se imprime en el notebook.

In [ ]:
%%bash
jq -r '.data.certificate' /tmp/vault-pki-unwrapped-cert.json > /tmp/vault-pki-unwrapped-cert.pem

openssl x509 -in /tmp/vault-pki-unwrapped-cert.pem \
  -noout \
  -subject \
  -issuer \
  -dates \
  -serial